# 5. Pipeline
Encapsulamos preprocesamiento y modelo para aceptar datos sin transformar.

In [ ]:
import pandas as pd
import pickle
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, QuantileTransformer, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

df = pd.read_csv('./data/titanic_clean.csv')
X = df.drop(columns=['Survived'])
y = df['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=100, stratify=y)
preprocessor = ColumnTransformer(transformers=[('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['Sex', 'Embarked']), ('age', QuantileTransformer(output_distribution='normal', random_state=100), ['Age']), ('fare', QuantileTransformer(output_distribution='normal', random_state=100), ['Fare'])], remainder='passthrough')
pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('scaler', MinMaxScaler()), ('model', LogisticRegression(max_iter=1000))])
parametros = {'model__C': [0.1, 1, 10], 'model__penalty': ['l1', 'l2'], 'model__solver': ['liblinear']}
grid_search = GridSearchCV(pipeline, parametros, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)
print(f'Precisión: {accuracy_score(y_test, grid_search.predict(X_test)):.2%}')
print('Mejores parámetros:', grid_search.best_params_)
with open('pipeline.pkl', 'wb') as archivo_estimador:
    pickle.dump(grid_search.best_estimator_, archivo_estimador)
print('Pipeline guardado en pipeline.pkl')